# 02 — Ablasi Backbone (3 Varian) & Aggregator B2 (Image-Only vs Multimodal)

Notebook ini menjalankan dua kelompok ablasi utama pada **5-fold pasien yang identik**:
1. **Ablasi Backbone (3 Varian):** `stage7_8` vs `stage7_only` vs `stage8_only` pada Multimodal FiLM Hierarchical.
2. **Ablasi Aggregator B2 (Target #2):** `mv_mean` vs `mv_lstm` vs `cross_attn` (Scaled Dot-Product Attention) untuk **Image-Only DAN Multimodal**, pada 3 formulasi tugas:
   - **Binary Diagnosis** (Appendicitis vs No Appendicitis)
   - **Binary Severity** (Complicated vs All, sesuai Table 6 Marcinkevičs et al., 2024)
   - **Hierarchical 3-Class** (No / Uncomplicated / Complicated)

In [1]:
from pathlib import Path
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from IPython.display import display

# Bisa dijalankan dari root proyek atau folder Proposed Model 3
candidates = [Path.cwd(), *Path.cwd().parents]
PM3_DIR = next(
    (p if (p / "pm3_data.py").is_file() else p / "Proposed Model 3")
    for p in candidates
    if (p / "pm3_data.py").is_file() or (p / "Proposed Model 3" / "pm3_data.py").is_file()
)
ROOT = PM3_DIR.parent
if str(PM3_DIR) not in sys.path:
    sys.path.insert(0, str(PM3_DIR))

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PM3_DIR:", PM3_DIR)
print("PyTorch:", torch.__version__, "| Device:", DEVICE)
if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

from pm3_data import USConfig, build_image_cache, images_by_patient, load_cohort, make_stratified_folds
from pm3_model import ModelConfig
from pm3_train import TrainConfig, run_experiment

RUN_BACKBONE_ABLATION = True   # Ubah ke True untuk menjalankan 5-fold ablasi 3 backbone
RUN_AGGREGATOR_IMAGE  = False   # Ubah ke True untuk menjalankan B2 Image-Only (Mean vs LSTM vs Cross-Attn)
RUN_AGGREGATOR_MULTI  = False   # Ubah ke True untuk menjalankan B2 Multimodal (Mean vs LSTM vs Cross-Attn)
SKIP_IF_DONE          = True    # Otomatis skip fold/run yang sudah selesai agar aman saat resume

cohort = load_cohort(ROOT / "app_data_filtered.csv")
manifest = pd.read_csv(ROOT / "image_manifest_filtered.csv")
manifest = manifest.loc[manifest["US_Number"].isin(cohort.index)].copy().reset_index(drop=True)
folds = make_stratified_folds(cohort, seed=1, n_splits=5)
bags_all = images_by_patient(manifest, bag_mode="all")
image_cache = build_image_cache(manifest, ROOT / "US_Pictures_clean", USConfig(), PM3_DIR / "cache")

BASE_TRAIN = TrainConfig(epochs=60, batch_size=16, patience=15, warmup_freeze_epochs=5, seed=1)

PM3_DIR: d:\Kuliah\Magister\Semester 1\Topik Dalam Pengolahan Citra Digital\Multimodal MIL-Attention based Hierarchical Classification Regensburg Pediatric Dataset\Proposed Model 3
PyTorch: 2.14.1+cu130 | Device: cuda
GPU: NVIDIA GeForce RTX 3060 Laptop GPU
Preprocessing/cache PM3: 250/1741
Preprocessing/cache PM3: 500/1741
Preprocessing/cache PM3: 750/1741
Preprocessing/cache PM3: 1000/1741
Preprocessing/cache PM3: 1250/1741
Preprocessing/cache PM3: 1500/1741


## 1. Ablasi Backbone EfficientNet-B0 (`stage7_8` vs `stage7_only` vs `stage8_only`)

In [2]:
backbone_rows = []
for bmode in ("stage7_8", "stage7_only", "stage8_only"):
    out_dir = PM3_DIR / "runs" / f"02_backbone_{bmode}"
    if RUN_BACKBONE_ABLATION:
        mcfg = ModelConfig(modality="multimodal", backbone_mode=bmode, aggregator="cross_attn", fusion_mode="film", task_mode="hierarchical")
        res = run_experiment(cohort, folds, bags_all, image_cache, mcfg, BASE_TRAIN, out_dir, DEVICE, skip_if_done=SKIP_IF_DONE, variant_label=f"backbone_{bmode}")
        cal = res["oof_calibrated"]
    elif (out_dir / "oof_metrics_calibrated.json").exists():
        cal = json.loads((out_dir / "oof_metrics_calibrated.json").read_text(encoding="utf-8"))
    else:
        continue
    backbone_rows.append({
        "backbone_mode": bmode,
        "accuracy": cal["accuracy"],
        "macro_f1": cal["macro_f1"],
        "multiclass_auroc": cal["multiclass_auroc"],
        "complication_recall": cal["complication_recall"],
    })
if backbone_rows:
    bb_df = pd.DataFrame(backbone_rows)
    bb_df.to_csv(PM3_DIR / "runs" / "02_backbone_ablation_summary.csv", index=False)
    display(bb_df.round(4))

[SKIP] Run sudah selesai, memuat hasil dari: d:\Kuliah\Magister\Semester 1\Topik Dalam Pengolahan Citra Digital\Multimodal MIL-Attention based Hierarchical Classification Regensburg Pediatric Dataset\Proposed Model 3\runs\02_backbone_stage7_8
[RESUME] Fold 1 (backbone_stage7_only) sudah selesai — memuat hasil tersimpan (val_auc=0.8395, val_f1=0.6837).
[RESUME] Fold 2 (backbone_stage7_only) sudah selesai — memuat hasil tersimpan (val_auc=0.7594, val_f1=0.5860).
[RESUME] Fold 3 (backbone_stage7_only) sudah selesai — memuat hasil tersimpan (val_auc=0.8090, val_f1=0.6234).
[RESUME] Fold 4 (backbone_stage7_only) sudah selesai — memuat hasil tersimpan (val_auc=0.8030, val_f1=0.5734).

backbone_stage7_only | fold 5 | epoch 1/60 | best epoch 1 | backbone=FROZEN(warmup) | 105.5s
TRAIN | Loss 2.581618 | Acc 0.4665 | F1 macro 0.4075 | AUROC 0.6005
  loss raw      : diag=0.690712 | comp=0.742064 | hier_nll=0.966232 | concept=0.872786 | aux_img=0.759913
  loss weighted : diag=0.690712 | comp=0.7420

,backbone_mode,accuracy,macro_f1,multiclass_auroc,complication_recall
0,stage7_8,0.6401,0.6222,0.7925,0.5208
1,stage7_only,0.6384,0.6291,0.8009,0.6042
2,stage8_only,0.6298,0.6150,0.7992,0.5521


## 2. Ablasi Aggregator B2: `mv_mean` vs `mv_lstm` vs `cross_attn` (Image-Only & Multimodal)

In [3]:
# Pilih backbone terbaik dari ablasi di atas (default: stage7_8)
BEST_BACKBONE = "stage7_8"

agg_rows = []
for modality, flag in (("image_only", RUN_AGGREGATOR_IMAGE), ("multimodal", RUN_AGGREGATOR_MULTI)):
    for task_mode in ("binary_diag", "binary_sev", "hierarchical"):
        for agg in ("mv_mean", "mv_lstm", "cross_attn"):
            out_dir = PM3_DIR / "runs" / f"02_b2_{modality}_{task_mode}_{agg}"
            if flag:
                mcfg = ModelConfig(
                    modality=modality,
                    backbone_mode=BEST_BACKBONE,
                    aggregator=agg,
                    fusion_mode="film",
                    task_mode=task_mode,
                )
                res = run_experiment(cohort, folds, bags_all, image_cache, mcfg, BASE_TRAIN, out_dir, DEVICE, skip_if_done=SKIP_IF_DONE, variant_label=f"{modality}_{task_mode}_{agg}")
                cal = res["oof_calibrated"]
            elif (out_dir / "oof_metrics_calibrated.json").exists():
                cal = json.loads((out_dir / "oof_metrics_calibrated.json").read_text(encoding="utf-8"))
            else:
                continue
            agg_rows.append({
                "modality": modality,
                "task_mode": task_mode,
                "aggregator": agg,
                "accuracy": cal["accuracy"],
                "macro_f1": cal["macro_f1"],
                "auroc": cal["primary_auroc"],
                "aupr": cal.get("aupr", cal.get("macro_aupr")),
            })

if agg_rows:
    agg_df = pd.DataFrame(agg_rows)
    agg_df.to_csv(PM3_DIR / "runs" / "02_b2_aggregator_summary.csv", index=False)
    display(agg_df.round(4))